In [3]:
import pandas as pd
import os


print('Loading the Master Dataset...')

df=pd.read_csv('../data/raw/collected_funds_data.csv')

df['date']=pd.to_datetime(df['date'])

print(f'Dataset completely loaded of length {len(df)} rows')

display(df.head())

Loading the Master Dataset...
Dataset completely loaded of length 81400 rows


,date,nav,scheme_code,scheme_name,plan_type
0,2019-01-01,158.04,118272,CANARA ROBECO EQUITY HYBRID FUND - DIRECT PLAN...,Direct
1,2019-01-02,157.37,118272,CANARA ROBECO EQUITY HYBRID FUND - DIRECT PLAN...,Direct
2,2019-01-03,156.53,118272,CANARA ROBECO EQUITY HYBRID FUND - DIRECT PLAN...,Direct
3,2019-01-04,156.74,118272,CANARA ROBECO EQUITY HYBRID FUND - DIRECT PLAN...,Direct
4,2019-01-07,157.17,118272,CANARA ROBECO EQUITY HYBRID FUND - DIRECT PLAN...,Direct


In [7]:
import pandas as pd

print('Cleaning fund names and pivoting data...')

# 1. Force names to uppercase and strip out the trailing plan text on the fly
df['scheme_name'] = df['scheme_name'].str.upper()

# Clean up Direct indicators
df['scheme_name'] = df['scheme_name'].str.replace(' - DIRECT PLAN - GROWTH', '', regex=False)
df['scheme_name'] = df['scheme_name'].str.replace(' DIRECT PLAN - GROWTH', '', regex=False)

# Clean up Regular indicators
df['scheme_name'] = df['scheme_name'].str.replace(' - REGULAR PLAN - GROWTH', '', regex=False)
df['scheme_name'] = df['scheme_name'].str.replace(' REGULAR PLAN - GROWTH', '', regex=False)

# Strip any accidental remaining edge spaces
df['scheme_name'] = df['scheme_name'].str.strip()

# 2. NOW run the pivot table with clean, matching names
processed_df = df.pivot_table(
    index=['date', 'scheme_name'],
    columns='plan_type',
    values='nav'
).reset_index()

# 3. Rename columns safely
processed_df.columns.name = None
processed_df = processed_df.rename(columns={'Direct': 'direct_nav', 'Regular': 'regular_nav'})

# 4. Now drop rows where pairs don't exist (it won't wipe the table out anymore!)
processed_df = processed_df.dropna(subset=['direct_nav', 'regular_nav'])

print(f'Success! Pivoted dataset length: {len(processed_df)} rows')
display(processed_df.head())




Cleaning fund names and pivoting data...
Success! Pivoted dataset length: 40700 rows


,date,scheme_name,direct_nav,regular_nav
0,2019-01-01,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.65,13.02
1,2019-01-01,ADITYA BIRLA SUN LIFE MANUFACTURING EQUITY FUND,13.84,13.33
2,2019-01-01,AXIS EQUITY ADVANTAGE FUND - SERIES 1 OPTION,11.54,11.24
3,2019-01-01,AXIS EQUITY ADVANTAGE FUND - SERIES 2 OPTION,11.06,10.77
4,2019-01-01,AXIS EQUITY SAVINGS FUND,12.93,12.42


In [8]:
# 1. Calculate the raw absolute difference in NAV
processed_df['nav_difference'] = processed_df['direct_nav'] - processed_df['regular_nav']

# 2. Calculate the percentage drag relative to the regular plan NAV
# This tells us the exact compounding premium the direct plan holds over regular
processed_df['pct_drag'] = (processed_df['nav_difference'] / processed_df['regular_nav']) * 100

print("✅ Target variables calculated successfully!")
display(processed_df[['date', 'scheme_name', 'direct_nav', 'regular_nav', 'nav_difference', 'pct_drag']].head())

✅ Target variables calculated successfully!


,date,scheme_name,direct_nav,regular_nav,nav_difference,pct_drag
0,2019-01-01,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.65,13.02,0.63,4.838710
1,2019-01-01,ADITYA BIRLA SUN LIFE MANUFACTURING EQUITY FUND,13.84,13.33,0.51,3.825956
2,2019-01-01,AXIS EQUITY ADVANTAGE FUND - SERIES 1 OPTION,11.54,11.24,0.30,2.669039
3,2019-01-01,AXIS EQUITY ADVANTAGE FUND - SERIES 2 OPTION,11.06,10.77,0.29,2.692665
4,2019-01-01,AXIS EQUITY SAVINGS FUND,12.93,12.42,0.51,4.106280


In [10]:
## Now the Market remains closed on satursdays and sundays so we will tackle that
processed_df=processed_df.sort_values(['scheme_name','date'])

fixed_prices=[]

for scheme_name , group_data in processed_df.groupby('scheme_name'):

    group_data=group_data.set_index('date') # .resample only works if index are dates

    group_data=group_data.resample('D').ffill() # It creates rows even for nan values and ffill is forward fill which tells to copy paste the value of previous rows

    group_data=group_data.reset_index()

    group_data['scheme_name']=scheme_name

    fixed_prices.append(group_data)

filled_df=pd.concat(fixed_prices , ignore_index=True)

filled_df['date']=pd.to_datetime(filled_df['date'])

print(f"Holiday cleanup complete! Expanded from 40,700 rows to {len(filled_df)} continuous rows.")
display(filled_df.head(10))



Holiday cleanup complete! Expanded from 40,700 rows to 61064 continuous rows.


,date,scheme_name,direct_nav,regular_nav,nav_difference,pct_drag
0,2019-01-01,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.65,13.02,0.63,4.838710
1,2019-01-02,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.61,12.99,0.62,4.772902
2,2019-01-03,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.55,12.93,0.62,4.795050
3,2019-01-04,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.55,12.93,0.62,4.795050
4,2019-01-05,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.55,12.93,0.62,4.795050
5,2019-01-06,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.55,12.93,0.62,4.795050
6,2019-01-07,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.56,12.94,0.62,4.791345
7,2019-01-08,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.55,12.93,0.62,4.795050
8,2019-01-09,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.57,12.95,0.62,4.787645
9,2019-01-10,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.57,12.95,0.62,4.787645


In [11]:
print("Calculating 7-day, 30-day, and 90-day returns for each fund...")

# 1. Create a container to hold our final calculations
final_calculated_pieces = []

# 2. Process one fund at a time to keep timelines independent
for scheme_name, group_data in filled_df.groupby('scheme_name'):
    
    # Sort chronologically to ensure historical math aligns perfectly
    group_data = group_data.sort_values('date')
    
    # 3. Calculate 7-Day Return
    # We look back 7 rows (since every calendar day now has a row)
    past_nav_7 = group_data['direct_nav'].shift(7)
    group_data['return_7d'] = ((group_data['direct_nav'] - past_nav_7) / past_nav_7) * 100
    
    # 4. Calculate 30-Day Return (1-month momentum)
    past_nav_30 = group_data['direct_nav'].shift(30)
    group_data['return_30d'] = ((group_data['direct_nav'] - past_nav_30) / past_nav_30) * 100
    
    # 5. Calculate 90-Day Return (3-month quarterly momentum)
    past_nav_90 = group_data['direct_nav'].shift(90)
    group_data['return_90d'] = ((group_data['direct_nav'] - past_nav_90) / past_nav_90) * 100
    
    # Save this completed piece
    final_calculated_pieces.append(group_data)

# 6. Merge everything back into our final master training table
final_features_df = pd.concat(final_calculated_pieces, ignore_index=True)

# 7. Drop the very first rows where lookbacks aren't possible (e.g., the first 90 days of 2019)
final_features_df = final_features_df.dropna(subset=['return_7d', 'return_30d', 'return_90d'])

print(f"Features successfully generated! Total clean data points: {len(final_features_df)} rows")
display(final_features_df[['date', 'scheme_name', 'pct_drag', 'return_7d', 'return_30d', 'return_90d']].head())

Calculating 7-day, 30-day, and 90-day returns for each fund...
Features successfully generated! Total clean data points: 57104 rows


,date,scheme_name,pct_drag,return_7d,return_30d,return_90d
90,2019-04-01,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,4.958678,1.526163,3.023599,2.344322
91,2019-04-02,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,4.940120,1.373825,3.392330,3.012491
92,2019-04-03,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,4.951238,0.937951,3.171091,3.247232
93,2019-04-04,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,4.954955,0.431034,2.342606,3.173432
94,2019-04-05,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,5.030030,0.143164,2.116788,3.247232


In [12]:
import os

print("Saving final engineered dataset...")

# 1. Define the file path (We are saving this in the 'processed' folder now, not 'raw')
save_path = '../data/processed/model_training_data.csv'

# 2. Save the table to a CSV file without the random index numbers
final_features_df.to_csv(save_path, index=False)

print(f"✅ Success! Your final dataset is securely saved at: {save_path}")

Saving final engineered dataset...
✅ Success! Your final dataset is securely saved at: ../data/processed/model_training_data.csv
